# 07: Q学習（テーブル型）による倒立振子制御

これまでのノートブックでは **方策探索**（ランダム探索・ヒルクライミング・進化戦略）を使ってきました。
これらは「パラメータを試行錯誤で最適化する」手法です。

このノートブックでは **Q学習** という全く異なるアプローチを学びます。

| アプローチ | 考え方 |
|-----------|--------|
| 方策探索 | 「どんなパラメータが良いか？」を試行錯誤で探す |
| **Q学習** | 「各状態で各行動がどれくらい良いか？」を学習する（価値ベース） |

## 学習内容
1. Q学習の理論（Bellman方程式・ε-greedy探索）
2. 連続状態空間の離散化（ビン化）
3. Q学習の実装と学習の可視化
4. 既存ポリシーとの性能比較

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.environments import InvertedPendulumEnv
from src.policies import RandomPolicy, LinearPolicy, QPolicy
from src.utils.discretizer import StateDiscretizer
from src.utils.training import train_q_learning, evaluate_policy

print('インポート完了！')

---
## 1. Q学習の理論

### Q値とは？

**Q値** Q(s, a) は「状態 s で行動 a を選んだときの**期待累積報酬**」です。

最適Q値は **Bellman方程式** を満たします：

$$Q^*(s, a) = r + \gamma \max_{a'} Q^*(s', a')$$

- $r$: 即時報酬
- $\gamma \in [0,1]$: 割引率（将来の報酬をどれくらい重視するか）
- $s'$: 次の状態

### TD更新式

実際の経験 $(s, a, r, s')$ をもとにQ値を少しずつ修正します：

$$Q(s, a) \leftarrow Q(s, a) + \alpha \underbrace{\left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]}_{\text{TD誤差}}$$

- $\alpha$: 学習率
- **TD誤差**：「予測していたQ値」と「実際の報酬 + 次状態での最大Q値」の差

### ε-greedy探索

$$\pi(s) = \begin{cases} \text{ランダム行動} & \text{確率 } \varepsilon \\ \arg\max_a Q(s, a) & \text{確率 } 1-\varepsilon \end{cases}$$

εはエピソードごとに減衰させ、探索から活用にシフトします（例: $\varepsilon \leftarrow \varepsilon \times 0.995$）。

---
## 2. 環境の確認

In [ ]:
env = InvertedPendulumEnv()
state = env.reset()

print('=== 環境の基本情報 ===')
print(f'状態の次元: [x, x_dot, theta, theta_dot]')
print(f'初期状態:    {state}')
print(f'力の上限:    ±{env.force_mag} N')
print(f'位置の上限:  ±{env.x_threshold} m')
print(f'角度の上限:  ±{env.theta_threshold:.4f} rad ({np.degrees(env.theta_threshold):.1f}°)')
print(f'最大ステップ: {env.max_steps}')

---
## 3. 状態の離散化

テーブル型Q学習は**離散的な状態空間**が必要です。
連続状態 [x, x_dot, theta, theta_dot] を整数インデックスに変換します。

| 次元 | 範囲 | ビン数 | 理由 |
|------|------|--------|------|
| x | ±2.4 m | 10 | 大まかな位置で十分 |
| x_dot | ±3.0 m/s | 10 | 符号と大きさが重要 |
| **theta** | ±0.21 rad | **20** | 制御の核心なので細かく分割 |
| theta_dot | ±3.0 rad/s | 10 | 符号が重要 |

**総状態数** = 10 × 10 × 20 × 10 = **20,000**

In [ ]:
disc = StateDiscretizer()
print(disc)
print(f'総状態数: {disc.n_states:,}')

# encode / decode の往復確認
test_state = np.array([0.0, 0.0, 0.05, 0.1])
idx = disc.encode(test_state)
center = disc.decode(idx)
print(f'\n元の状態:   {test_state}')
print(f'エンコード: {idx}')
print(f'ビン中心:   {np.round(center, 4)}')

In [ ]:
# theta × theta_dot の2Dグリッド可視化
theta_n = disc.bins_per_dim[2]
tdot_n  = disc.bins_per_dim[3]

grid = np.array([
    [np.ravel_multi_index([5, 5, ti, td], disc.bins_per_dim)
     for td in range(tdot_n)]
    for ti in range(theta_n)
])

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(grid, origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(im, ax=ax, label='状態インデックス')
ax.set_xlabel('theta_dot ビン')
ax.set_ylabel('theta ビン')
ax.set_title('状態空間のビングリッド（x, x_dot を中央固定）')
plt.tight_layout()
plt.show()

---
## 4. QPolicy の構築

In [ ]:
policy = QPolicy(
    state_discretizer=disc,
    n_actions=11,         # -10, -8, ..., 0, ..., 8, 10 N
    alpha=0.1,            # 学習率
    gamma=0.99,           # 割引率
    epsilon=1.0,          # 初期探索率（最初は完全ランダム）
    epsilon_min=0.01,
    epsilon_decay=0.995,
)

print(policy)
print(f'Qテーブルサイズ: {policy.q_table.table.shape}')
print(f'行動の値: {policy.action_values}')

In [ ]:
# ε=1.0 時の行動分布（ほぼ均一なランダム）
state = env.reset()
sampled_actions = [policy.get_action(state) for _ in range(300)]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(sampled_actions, bins=policy.n_actions, edgecolor='white', color='steelblue')
ax.set_xlabel('行動（力 N）')
ax.set_ylabel('回数')
ax.set_title(f'ε={policy.epsilon:.1f} での行動分布（300回）→ 均一なランダム')
plt.tight_layout()
plt.show()

---
## 5. Q学習の実行

In [ ]:
np.random.seed(42)

result = train_q_learning(
    env=env,
    policy=policy,
    n_episodes=2000,
    eval_interval=200,
    eval_episodes=10,
    verbose=True,
)

print(f'\n学習完了！')
print(f'最終 ε:       {policy.epsilon:.4f}')
print(f'最終 eval 報酬: {result["eval_rewards"][-1]:.1f} / {env.max_steps}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 学習曲線（移動平均）
rewards = np.array(result['episode_rewards'])
window = 50
ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
axes[0].plot(rewards, alpha=0.3, color='steelblue')
axes[0].plot(range(window - 1, len(rewards)), ma, color='steelblue', lw=2, label=f'{window}話移動平均')
axes[0].set_xlabel('エピソード')
axes[0].set_ylabel('総報酬')
axes[0].set_title('学習曲線')
axes[0].legend()

# ε の減衰
axes[1].plot(result['epsilon_history'], color='orange')
axes[1].axhline(policy.epsilon_min, ls='--', color='gray', label=f'最小値 ({policy.epsilon_min})')
axes[1].set_xlabel('エピソード')
axes[1].set_ylabel('ε')
axes[1].set_title('ε-greedy 探索率の減衰')
axes[1].legend()

# 評価報酬の推移
axes[2].plot(result['eval_episodes'], result['eval_rewards'], 'o-', color='green')
axes[2].axhline(env.max_steps, ls='--', color='gray', label=f'最大値 ({env.max_steps})')
axes[2].set_xlabel('エピソード')
axes[2].set_ylabel('Greedy 評価報酬')
axes[2].set_title('評価報酬の推移')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 6. Q テーブルの可視化

In [ ]:
theta_n = disc.bins_per_dim[2]
tdot_n  = disc.bins_per_dim[3]

# x, x_dot を中央ビンで固定し、theta × theta_dot のQ値を可視化
cx, cxd = disc.bins_per_dim[0] // 2, disc.bins_per_dim[1] // 2

max_q_grid    = np.zeros((theta_n, tdot_n))
greedy_a_grid = np.zeros((theta_n, tdot_n))

for ti in range(theta_n):
    for td in range(tdot_n):
        sidx = np.ravel_multi_index([cx, cxd, ti, td], disc.bins_per_dim)
        max_q_grid[ti, td]    = policy.q_table.max_value(sidx)
        greedy_a_grid[ti, td] = policy.q_table.greedy_action(sidx)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(max_q_grid, origin='lower', aspect='auto', cmap='RdYlGn')
plt.colorbar(im0, ax=axes[0], label='max Q(s, a)')
axes[0].set_xlabel('theta_dot ビン')
axes[0].set_ylabel('theta ビン')
axes[0].set_title('最大Q値マップ（高い = 良い状態）')

im1 = axes[1].imshow(greedy_a_grid, origin='lower', aspect='auto', cmap='coolwarm')
plt.colorbar(im1, ax=axes[1], label='greedy action index')
axes[1].set_xlabel('theta_dot ビン')
axes[1].set_ylabel('theta ビン')
axes[1].set_title('最適行動マップ（赤=右押し, 青=左押し）')

plt.tight_layout()
plt.show()
print('theta が正（右傾き）のとき右に力を入れる（赤）パターンが見えると直感的に正しい')

---
## 7. 既存ポリシーとの性能比較

In [ ]:
# 評価は greedy モード（ε=0）で
policy.epsilon = 0.0

policies_to_compare = {
    'Random':         RandomPolicy(),
    'Linear (手動)':  LinearPolicy(weights=np.array([0, 0, 10, 3])),
    'Q-Learning':     policy,
}

comp_results = {}
for name, p in policies_to_compare.items():
    r = evaluate_policy(env, p, n_episodes=50, seed=42)
    comp_results[name] = r
    print(f'{name:20s}: 平均 {r["mean_reward"]:6.1f} ± {r["std_reward"]:5.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71']
labels = list(comp_results.keys())
data   = [comp_results[k]['episode_rewards'] for k in labels]

bp = ax.boxplot(data, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('総報酬（生存ステップ数）', fontsize=12)
ax.set_title('ポリシー別 性能比較（50エピソード）', fontsize=14)
ax.axhline(env.max_steps, ls='--', color='gray', alpha=0.6, label=f'最大値 ({env.max_steps})')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. まとめと次のステップ

### 学んだこと

| 概念 | 内容 |
|------|------|
| **Q値** | 状態-行動ペアの期待累積報酬 |
| **Bellman方程式** | Q値の再帰的定義 |
| **TD更新** | 1ステップの経験でQ値をオンライン更新 |
| **ε-greedy** | 探索（ランダム）と活用（greedy）のバランス |
| **状態離散化** | 連続空間をテーブルで扱うためのビン化 |

### テーブル型Q学習の限界

- 状態空間が大きいとQテーブルが爆発する（**次元の呪い**）
- 今回は 20,000 状態でしたが、より複雑な環境では数百万〜無限になる
- → 解決策：**DQN**（Deep Q-Network）でQテーブルをニューラルネットで近似

### 次のノートブック

**08_policy_gradients.ipynb** — REINFORCE（方策勾配法）
- Q学習と異なり、価値関数を使わず方策を直接最適化する
- 連続行動空間にそのまま対応できる